In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path

def check_directory(dir_path):
    path = Path(dir_path).resolve()
    print(f"\nChecking directory: {path}")
    print(f"Directory exists: {path.exists()}")
    if path.exists():
        print("Contents:")
        for item in path.iterdir():
            print(f"  {item.name}")
    return path

def analyze_raw_image(filepath, expected_width, expected_height):
    """Analyze a raw YUV file to confirm format and size"""
    filesize = os.path.getsize(filepath)
    # YUV422 uses 2 bytes per pixel
    expected_size = expected_width * expected_height * 2
    
    if filesize != expected_size:
        print(f"Warning: File size {filesize} doesn't match expected size {expected_size}")
        return None
    
    with open(filepath, 'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
        # Reshape to get UYVY pattern
        data = data.reshape((expected_height, expected_width * 2))
    
    return data

def jpeg_to_uyvy(jpg_path, target_width, target_height):
    """Convert JPEG to UYVY format"""
    img = cv2.imread(jpg_path)
    if img is None:
        print(f"Failed to read {jpg_path}")
        return None
    
    aspect = img.shape[1] / img.shape[0]
    target_aspect = target_width / target_height
    
    if aspect > target_aspect:
        new_width = int(target_height * aspect)
        img = cv2.resize(img, (new_width, target_height))
        start = (new_width - target_width) // 2
        img = img[:, start:start+target_width]
    else:
        new_height = int(target_width / aspect)
        img = cv2.resize(img, (target_width, new_height))
        start = (new_height - target_height) // 2
        img = img[start:start+target_height, :]

    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    uyvy = np.zeros((target_height, target_width * 2), dtype=np.uint8)
    
    uyvy[:, 1::2] = yuv[:, :, 0]  # Y values
    
    u = yuv[:, :, 1]
    v = yuv[:, :, 2]
    
    u_sub = u[:, ::2]
    v_sub = v[:, ::2]
    
    uyvy[:, 0::4] = u_sub
    uyvy[:, 2::4] = v_sub
    
    return uyvy

def process_directory(jpg_dir, output_dir, target_width, target_height):
    """Process all JPEGs in a directory"""
    jpg_dir = Path(jpg_dir).resolve()
    print(f"Processing JPEGs from: {jpg_dir}")
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    for jpg_file in jpg_dir.glob("*.jpg"):
        uyvy_data = jpeg_to_uyvy(str(jpg_file), target_width, target_height)
        if uyvy_data is not None:
            output_path = output_dir / (jpg_file.stem + '.raw')  # Changed to .raw
            uyvy_data.tofile(str(output_path))
            print(f"Converted {jpg_file} to {output_path}")

def analyze_raw_directory(raw_dir, width, height):
    """Analyze all raw files in a directory"""
    raw_dir = Path(raw_dir)
    for raw_file in raw_dir.glob("*.raw"):  # Changed to .raw
        print(f"Analyzing {raw_file}...")
        data = analyze_raw_image(str(raw_file), width, height)
        if data is not None:
            print(f"File matches expected format: {width}x{height} UYVY")

raw_dir = check_directory("./drone pictures")
jpg_dir = check_directory("./verified_images/unlabeled")
output_dir = Path("./verified_images_raw/unlabeled")

# Create output directory if it doesn't exist
output_dir.parent.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print("\nStarting raw file analysis...")
analyze_raw_directory(raw_dir, 240, 240)  # Adjust width/height as needed

print("\nStarting JPEG conversion...")
process_directory(jpg_dir, output_dir, 120, 120)  # Adjust width/height as needed


Checking directory: /home/daniel/Documents/GitHub/paparazzi/BottomCamDetector/drone pictures
Directory exists: True
Contents:
  frame_1285_yuv.raw
  frame_1385_yuv.raw
  frame_1150_yuv.raw
  frame_660_yuv.raw
  frame_545_yuv.raw
  frame_430_yuv.raw
  frame_950_yuv.raw
  frame_1105_yuv.raw
  frame_845_yuv.raw
  frame_910_yuv.raw
  frame_980_yuv.raw
  frame_1255_yuv.raw
  frame_1200_yuv.raw
  frame_920_yuv.raw
  frame_855_yuv.raw
  frame_960_yuv.raw
  frame_375_yuv.raw
  frame_1045_yuv.raw
  frame_620_yuv.raw
  frame_775_yuv.raw
  frame_870_yuv.raw
  frame_1145_yuv.raw
  frame_1215_yuv.raw
  frame_1115_yuv.raw
  frame_1370_yuv.raw
  frame_790_yuv.raw
  frame_1180_yuv.raw
  frame_1315_yuv.raw
  frame_1155_yuv.raw
  frame_1160_yuv.raw
  frame_690_yuv.raw
  frame_520_yuv.raw
  frame_975_yuv.raw
  frame_895_yuv.raw
  frame_330_yuv.raw
  frame_1205_yuv.raw
  frame_1265_yuv.raw
  frame_440_yuv.raw
  frame_1365_yuv.raw
  frame_820_yuv.raw
  frame_755_yuv.raw
  frame_860_yuv.raw
  frame_770_yuv